# Exportable ArXiv Recommender
This notebook trains a hybrid recommender and saves all required artifacts so the model can be reused outside this runtime (for example, after downloading from Colab).

In [1]:
import os
import json
import re
import time
import numpy as np
import joblib
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def stream_arxiv(file_path, limit=None):
    papers = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                continue

            papers.append({
                'id': data.get('id'),
                'title': data.get('title') or '',
                'abstract': data.get('abstract') or '',
                'categories': data.get('categories') or '',
                'authors': data.get('authors'),
                'update_date': data.get('update_date')
            })
    return papers

def clean_text(text):
    text = (text or '').lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text

def normalize(x):
    x = np.asarray(x, dtype=float)
    spread = x.max() - x.min()
    if spread == 0:
        return np.zeros_like(x)
    return (x - x.min()) / spread

In [3]:
def train_recommender(data_file, limit=None, random_seed=42, large_dataset_mode=True):
    t0 = time.time()
    papers = stream_arxiv(data_file, limit=limit)
    for p in papers:
        p['text'] = clean_text(p['title'] + ' ' + p['abstract'])

    corpus = [p['text'] for p in papers]

    if large_dataset_mode:
        vectorizer = TfidfVectorizer(
            stop_words='english',
            ngram_range=(1, 1),
            min_df=5,
            max_features=200000,
            sublinear_tf=True,
            dtype=np.float32
        )
    else:
        vectorizer = TfidfVectorizer(
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )

    tfidf_matrix = vectorizer.fit_transform(corpus).astype(np.float32)

    rng = np.random.default_rng(random_seed)
    simulated_interactions = rng.integers(
        0, len(papers), size=min(50000, max(5000, len(papers) // 2))
    )
    popularity = np.bincount(simulated_interactions, minlength=len(papers)).astype(np.float32)

    artifacts = {
        'papers': papers,
        'vectorizer': vectorizer,
        'tfidf_matrix': tfidf_matrix,
        'popularity': popularity,
        'config': {
            'content_weight': 0.7,
            'popularity_weight': 0.3,
            'random_seed': random_seed,
            'large_dataset_mode': large_dataset_mode,
            'vectorizer_settings': {
                'ngram_range': list(vectorizer.ngram_range),
                'min_df': vectorizer.min_df,
                'max_features': vectorizer.max_features
            }
        }
    }

    elapsed_min = (time.time() - t0) / 60
    print(f'Training completed in {elapsed_min:.2f} minutes.')
    return artifacts

In [4]:
def save_artifacts(bundle_dir, artifacts):
    os.makedirs(bundle_dir, exist_ok=True)

    joblib.dump(artifacts['vectorizer'], os.path.join(bundle_dir, 'vectorizer.joblib'))
    sparse.save_npz(os.path.join(bundle_dir, 'tfidf_matrix.npz'), artifacts['tfidf_matrix'])
    np.save(os.path.join(bundle_dir, 'popularity.npy'), artifacts['popularity'])

    with open(os.path.join(bundle_dir, 'papers.json'), 'w', encoding='utf-8') as f:
        json.dump(artifacts['papers'], f, ensure_ascii=False)

    with open(os.path.join(bundle_dir, 'config.json'), 'w', encoding='utf-8') as f:
        json.dump(artifacts['config'], f, indent=2)

    print('Artifacts saved to:', bundle_dir)

In [5]:
def load_artifacts(bundle_dir):
    vectorizer = joblib.load(os.path.join(bundle_dir, 'vectorizer.joblib'))
    tfidf_matrix = sparse.load_npz(os.path.join(bundle_dir, 'tfidf_matrix.npz'))
    popularity = np.load(os.path.join(bundle_dir, 'popularity.npy'))

    with open(os.path.join(bundle_dir, 'papers.json'), 'r', encoding='utf-8') as f:
        papers = json.load(f)

    with open(os.path.join(bundle_dir, 'config.json'), 'r', encoding='utf-8') as f:
        config = json.load(f)

    return {
        'papers': papers,
        'vectorizer': vectorizer,
        'tfidf_matrix': tfidf_matrix,
        'popularity': popularity,
        'config': config
    }

In [6]:
def build_user_profile(tfidf_matrix, user_indices):
    return np.asarray(tfidf_matrix[user_indices].mean(axis=0))

def hybrid_recommend(artifacts, user_indices, top_k=10):
    tfidf_matrix = artifacts['tfidf_matrix']
    popularity = artifacts['popularity']
    w_content = artifacts['config']['content_weight']
    w_pop = artifacts['config']['popularity_weight']

    user_profile = build_user_profile(tfidf_matrix, user_indices)
    content_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    content_scores = normalize(content_scores)

    pop_scores = normalize(popularity)
    final_scores = normalize(w_content * content_scores + w_pop * pop_scores)

    final_scores[user_indices] = -1.0
    top_indices = final_scores.argsort()[-top_k:][::-1]

    return top_indices, {
        'content': content_scores,
        'popularity': pop_scores,
        'final': final_scores
    }

In [7]:
# Example: train and export (recommended defaults for large datasets)
DATA_FILE = '/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json'
BUNDLE_DIR = 'recommender_bundle'
LARGE_DATASET_MODE = True  # safer for Colab/Kaggle memory
LIMIT = None  # use full dataset

artifacts = train_recommender(
    DATA_FILE,
    limit=LIMIT,
    large_dataset_mode=LARGE_DATASET_MODE
    )
save_artifacts(BUNDLE_DIR, artifacts)
print('Papers trained on:', len(artifacts['papers']))
print('Large dataset mode:', artifacts['config']['large_dataset_mode'])
print('Vectorizer settings:', artifacts['config']['vectorizer_settings'])

Training completed in 6.93 minutes.
Artifacts saved to: recommender_bundle
Papers trained on: 3021763
Large dataset mode: True
Vectorizer settings: {'ngram_range': [1, 1], 'min_df': 5, 'max_features': 200000}


In [ ]:
# Example: load exported artifacts and run inference
loaded = load_artifacts(BUNDLE_DIR)
user_history = [1, 10, 25, 40]
top_indices, score_components = hybrid_recommend(loaded, user_history, top_k=5)

for idx in top_indices:
    title = loaded['papers'][int(idx)]['title']
    score = score_components['final'][int(idx)]
    print('Title:', title[:180])
    print('Relevance score (0-1):', round(float(score), 4))
    print()

Title: Placeholder Substructures III: A Bit-String-Driven ''Recipe Theory'' for
  Infinite-Dimensional Zero-Divisor Spaces
Relevance score (0-1): 0.9169

Title: Placeholder Substructures I: The Road from NKS to Scale-Free Networks Is
  Paved with Zero-Divisors
Relevance score (0-1): 0.8804

Title: On the computation of algebraic modular forms on compact inner forms of
  $\mathrm{GSp}_4$
Relevance score (0-1): 0.5341

Title: A higher Kac-Moody extension for two-dimensional gauge groups
Relevance score (0-1): 0.5174

Title: On the lifting of Hilbert cusp forms to Hilbert-Siegel cusp forms
Relevance score (0-1): 0.5007



## Colab Transfer Notes
- Save the notebook and download both the notebook file and the recommender_bundle folder.
- In Colab, you can also save recommender_bundle to Google Drive for persistence.
- On a new runtime or machine, only loading the bundle is needed for inference.

In [9]:
import shutil
import os

# Copy the entire recommender_bundle to a compressed file
shutil.make_archive('/kaggle/working/recommender_bundle', 'zip', '/kaggle/working', 'recommender_bundle')
print("Bundle compressed successfully!")

# List the files created
os.system('ls -lh /kaggle/working/*.zip')

Bundle compressed successfully!
-rw-r--r-- 1 root root 2.9G Apr 26 14:41 /kaggle/working/recommender_bundle.zip


0

In [10]:
import os
from IPython.display import FileLink

# Ensure you are in the working directory
os.chdir(r'/kaggle/working')

# Replace with your exact zip file name
FileLink(r'recommender_bundle.zip')

/kaggle/working/recommender_bundle.zip